# Step 8b: Full Comparison — Köppen Panels vs Data-Driven Clusters
### DI 501 Term Project | Eda Yilmaz

**Goal:**  
Directly compare all model results (Naive, DT, RF) and Wilcoxon significance between:
- Theory-based Köppen-Geiger panels (9 European countries, Step 4b)
- Data-driven k-means clusters (36 countries, Step 8)

---

In [22]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── Köppen Panel Results (from Step 4b) ───────────────────────────────────────
koppen = {
    "Mediterranean": {"n": 3304, "naive": 17.2, "dt": 17.3, "rf": 17.4, "sig_dt": False, "sig_rf": False},
    "Temperate":     {"n": 5851, "naive": 13.4, "dt": 12.2, "rf": 12.2, "sig_dt": True,  "sig_rf": True},
    "Continental":   {"n": 1507, "naive": 24.0, "dt": 23.0, "rf": 22.9, "sig_dt": True,  "sig_rf": True},
}

# ── Data-Driven Cluster Results (from Step 8) ─────────────────────────────────
clusters_data = [
    {"name": "Cluster 0 (Semi-Arid)",  "n": 8754,  "naive": 25.3, "dt": 24.1, "rf": 23.9, "sig_dt": True,  "sig_rf": True},
    {"name": "Cluster 1 (Temperate)",  "n": 20143, "naive": 18.7, "dt": 18.4, "rf": 18.4, "sig_dt": True,  "sig_rf": True},
    {"name": "Cluster 2 (Tropical)",   "n": 1381,  "naive": 31.6, "dt": 33.1, "rf": 33.3, "sig_dt": False, "sig_rf": False},
]

print("=== KOPPEN PANELS (9 European Countries) ===")
print(f"  {'Panel':<18} {'N':>6} {'Naive':>8} {'DT':>8} {'RF':>8}  {'DT Wilcoxon':>15} {'RF Wilcoxon':>15}")
print("-" * 82)
for p, v in koppen.items():
    s_dt = "SIGNIFICANT *" if v["sig_dt"] else "not sig."
    s_rf = "SIGNIFICANT *" if v["sig_rf"] else "not sig."
    print(f"  {p:<18} {v['n']:>6} {v['naive']:>7.1f}% {v['dt']:>7.1f}% {v['rf']:>7.1f}%  {s_dt:>15} {s_rf:>15}")

print()
print("=== DATA-DRIVEN CLUSTERS (36 Countries) ===")
print(f"  {'Cluster':<24} {'N':>6} {'Naive':>8} {'DT':>8} {'RF':>8}  {'DT Wilcoxon':>15} {'RF Wilcoxon':>15}")
print("-" * 88)
for v in clusters_data:
    s_dt = "SIGNIFICANT *" if v["sig_dt"] else "not sig."
    s_rf = "SIGNIFICANT *" if v["sig_rf"] else "not sig."
    print(f"  {v['name']:<24} {v['n']:>6} {v['naive']:>7.1f}% {v['dt']:>7.1f}% {v['rf']:>7.1f}%  {s_dt:>15} {s_rf:>15}")

print()
print("KEY DIFFERENCES:")
sig_k = sum(1 for v in koppen.values() if v["sig_rf"])
sig_c = sum(1 for v in clusters_data if v["sig_rf"])
print(f"  Coverage    : 9 European countries vs 36 global countries")
print(f"  RF sig.     : {sig_k}/3 panels vs {sig_c}/3 clusters")
print(f"  Med. RF worse than naive (17.4% vs 17.2%) — theory-based grouping misfires on Mediterranean")
print(f"  Data-driven clusters defined by same variables used in modelling")


=== KOPPEN PANELS (9 European Countries) ===
  Panel                   N    Naive       DT       RF      DT Wilcoxon     RF Wilcoxon
----------------------------------------------------------------------------------
  Mediterranean        3304    17.2%    17.3%    17.4%         not sig.        not sig.
  Temperate            5851    13.4%    12.2%    12.2%    SIGNIFICANT *   SIGNIFICANT *
  Continental          1507    24.0%    23.0%    22.9%    SIGNIFICANT *   SIGNIFICANT *

=== DATA-DRIVEN CLUSTERS (36 Countries) ===
  Cluster                       N    Naive       DT       RF      DT Wilcoxon     RF Wilcoxon
----------------------------------------------------------------------------------------
  Cluster 0 (Semi-Arid)      8754    25.3%    24.1%    23.9%    SIGNIFICANT *   SIGNIFICANT *
  Cluster 1 (Temperate)     20143    18.7%    18.4%    18.4%    SIGNIFICANT *   SIGNIFICANT *
  Cluster 2 (Tropical)       1381    31.6%    33.1%    33.3%         not sig.        not sig.

KEY DIFFE

---
## Comparison Figure

Colors: **Gray** = Naive Baseline | **Light Pink** = DT Tuned | **Blue** = RF Tuned  
Hatching (`////`) marks statistically significant improvement over naive (p < 0.05, Wilcoxon signed-rank).

In [23]:
# ── Colors ───────────────────────────────────────────────────────────────────
C_NAIVE = '#aaaaaa'   # gray
C_DT    = '#ffb3c6'   # light pink
C_RF    = '#5b9bd5'   # medium blue

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
title_str = 'Model Performance: Koppen Panels vs Data-Driven Clusters\n(Gray = Naive | Pink = DT Tuned | Blue = RF Tuned)'
fig.suptitle(title_str, fontsize=14, fontweight='bold')

legend_els = [
    Patch(facecolor=C_NAIVE, edgecolor='black', label='Naive Baseline'),
    Patch(facecolor=C_DT,    edgecolor='black', hatch='////', label='DT Tuned (significant, p<0.05)'),
    Patch(facecolor=C_DT,    edgecolor='black', label='DT Tuned (not significant)'),
    Patch(facecolor=C_RF,    edgecolor='black', hatch='////', label='RF Tuned (significant, p<0.05)'),
    Patch(facecolor=C_RF,    edgecolor='black', label='RF Tuned (not significant)'),
]

w = 0.22

# ── Left: Koppen panels ───────────────────────────────────────────────────────
ax = axes[0]
k_names = list(koppen.keys())
k_data  = [koppen[p] for p in k_names]
x = np.arange(len(k_names))

ax.bar(x - w, [d['naive'] for d in k_data], w, color=C_NAIVE, edgecolor='black', linewidth=0.7)
for i, d in enumerate(k_data):
    ax.bar(x[i],     d['dt'], w, color=C_DT, edgecolor='black', linewidth=0.7, hatch='////' if d['sig_dt'] else '')
    ax.bar(x[i] + w, d['rf'], w, color=C_RF, edgecolor='black', linewidth=0.7, hatch='////' if d['sig_rf'] else '')

for i, d in enumerate(k_data):
    ax.text(x[i]-w, d['naive']+0.4, f"{d['naive']:.1f}%", ha='center', fontsize=10, color='#444')
    col_dt = '#2a7a2a' if d['dt'] < d['naive'] else '#cc3333'
    col_rf = '#2a7a2a' if d['rf'] < d['naive'] else '#cc3333'
    ax.text(x[i],   d['dt']+0.4, f"{d['dt']:.1f}%", ha='center', fontsize=10, color=col_dt, fontweight='bold')
    ax.text(x[i]+w, d['rf']+0.4, f"{d['rf']:.1f}%", ha='center', fontsize=10, color=col_rf, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(k_names, fontsize=11)
ax.set_ylabel('rRMSE (%)', fontsize=11)
ax.set_title('Theory-Based Panels\n9 European Countries | 3 Koppen Panels', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 34)
ax.tick_params(axis='y', labelsize=10)

# ── Right: Data-driven clusters ───────────────────────────────────────────────
ax = axes[1]
x2 = np.arange(len(clusters_data))

ax.bar(x2 - w, [d['naive'] for d in clusters_data], w, color=C_NAIVE, edgecolor='black', linewidth=0.7)
for i, d in enumerate(clusters_data):
    ax.bar(x2[i],     d['dt'], w, color=C_DT, edgecolor='black', linewidth=0.7, hatch='////' if d['sig_dt'] else '')
    ax.bar(x2[i] + w, d['rf'], w, color=C_RF, edgecolor='black', linewidth=0.7, hatch='////' if d['sig_rf'] else '')

for i, d in enumerate(clusters_data):
    ax.text(x2[i]-w, d['naive']+0.4, f"{d['naive']:.1f}%", ha='center', fontsize=10, color='#444')
    col_dt = '#2a7a2a' if d['dt'] < d['naive'] else '#cc3333'
    col_rf = '#2a7a2a' if d['rf'] < d['naive'] else '#cc3333'
    ax.text(x2[i],   d['dt']+0.4, f"{d['dt']:.1f}%", ha='center', fontsize=10, color=col_dt, fontweight='bold')
    ax.text(x2[i]+w, d['rf']+0.4, f"{d['rf']:.1f}%", ha='center', fontsize=10, color=col_rf, fontweight='bold')

ax.set_xticks(x2)
ax.set_xticklabels([d['name'] for d in clusters_data], fontsize=11)
ax.set_ylabel('rRMSE (%)', fontsize=11)
ax.set_title('Data-Driven Clusters\n36 Countries | 3 K-Means Clusters', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 45)
ax.tick_params(axis='y', labelsize=10)
ax.text(0.99, 0.02, '* p < 0.05 Wilcoxon signed-rank test',
        transform=ax.transAxes, ha='right', fontsize=9, style='italic')

# ── Shared legend centered below both panels ──────────────────────────────────
fig.legend(handles=legend_els, loc='lower center', ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.04), frameon=True, edgecolor='#cccccc')

plt.tight_layout(rect=[0, 0.08, 1, 1])
out = './fig_koppen_vs_clusters_comparison.png'
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: ./fig_koppen_vs_clusters_comparison.png


/var/folders/g8/1hlvtqbj25qb2y3hhh_f3dvh0000gn/T/ipykernel_13285/2571394217.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
